# ITS Predict

In [1]:
%pip install -qq --ignore-requires-python --no-deps 'graphies[predict] @ git+https://github.com/lukasmki/graphies.git'
%pip install -qq pydantic networkx datasets polars torch

import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
else:
    print(
        "CUDA is not available. Please ensure you have selected a GPU runtime in 'Runtime > Change runtime type'."
    )

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
Device name: NVIDIA A100-SXM4-40GB


## Setup Data Sources

In [2]:
!mkdir -p its-data
!wget -nv https://raw.githubusercontent.com/lukasmki/graphies-applications/refs/heads/main/grammars/its.json -O its-data/its.json

from pathlib import Path

GRAMMAR_PATH = next(Path().glob("its-data/*.json"))

2026-04-18 01:14:27 URL:https://raw.githubusercontent.com/lukasmki/graphies-applications/refs/heads/main/grammars/its.json [8747/8747] -> "its-data/its.json" [1]


In [ ]:
from datasets import load_dataset

hf_dataset = load_dataset("lukaskim/MappedCRD", "its", split="train")

## Setup Data Loaders

In [5]:
from graphies.predict import GraphiesTokenizer, HFGraphiesDataset
from torch.utils.data import DataLoader, random_split

tokenizer = GraphiesTokenizer(GRAMMAR_PATH)
dataset = HFGraphiesDataset(
    hf_dataset, column="its_graphies", tokenizer=tokenizer, split=None
)

trn, tst = random_split(dataset, [0.9, 0.1])
torch.save(
    {"trn_indices": trn.indices, "tst_indices": tst.indices}, "its-data/split.pt"
)
trn_loader = DataLoader(
    dataset=trn,
    batch_size=256,
    shuffle=True,
    collate_fn=tokenizer.collate,
)
tst_loader = DataLoader(
    dataset=tst,
    batch_size=256,
    shuffle=False,
    collate_fn=tokenizer.collate,
)

## Trainer

In [6]:
from graphies.predict import GraphiesTrainer
from graphies.predict.models import GRU
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GRU(vocab_size=tokenizer.vocab_size)
optimizer = Adam(params=model.parameters(), lr=1e-3)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.1)

# include kwargs for classes to restart from checkpoitn
checkpoint = {
    "model_kwargs": {"vocab_size": tokenizer.vocab_size},
    "optimizer_kwargs": {"lr": 1e-3},
    "scheduler_kwargs": {"mode": "min", "patience": 3, "factor": 0.1},
}
trainer = GraphiesTrainer(model, optimizer, scheduler, device, checkpoint)

In [7]:
trainer.train(
    train=trn_loader,
    epochs=2,
    test=tst_loader,
    test_interval=1,
    log="its-data/log.csv",
    log_interval=1,
    checkpoint="its-data/chk.pt",
    checkpoint_interval=1,
)
trainer.save_model("its-data/model.pt")

Test 2: 100%|██████████| 549/549 [00:48<00:00, 11.26it/s, loss=1.0154]


In [8]:
# export to google drive
from google.colab import drive

drive.mount("/content/drive")
!zip -r its-data.zip its-data/
!cp its-data.zip '/content/drive/MyDrive/'

Mounted at /content/drive
  adding: its-data/ (stored 0%)
  adding: its-data/0-chk.pt (deflated 31%)
  adding: its-data/log.csv (deflated 26%)
  adding: its-data/1-chk.pt (deflated 31%)
  adding: its-data/model.pt (deflated 7%)
  adding: its-data/its.json (deflated 89%)
  adding: its-data/split.pt (deflated 31%)


# Run Inference

In [10]:
from graphies.predict import GraphiesModel

model = GraphiesModel.from_checkpoint(
    checkpoint="its-data/model.pt",
    tokenizer=tokenizer,
    model_cls=GRU,
    device=device,
)
sequences = model.generate(num=512, temperature=0.9, top_p=0.95, max_len=5000)
ref_sequences = hf_dataset[tst.indices]["its_graphies"]

In [11]:
sequences

['[BEGIN][-0Fork1][7][-0Si][-Fork1][0][-CH3][-Fork1][0][-CH3][-CH3][-C][=CH1][-C][-Fork1][0][-Cl][=C][-Fork1][3][-CH1][=CH1][-Link1][5][-Cl][END]',
 '[BEGIN][-C][=C][-Fork1][5][-CH1][=CH1][-CH1][=CH1][-Link1][4][-CH2][-CH1][-Fork2][1][1][-O][-C][0-Fork1][a][0-OH1][-C][=CH1][-CH1][=CH1][-CH1][=C][-Link1][4][-CH2][-CH3][=Fork1][0][=O][-0OH1][-Link2][1][5][-C][-Fork1][0][-NH2][=O][END]',
 '[BEGIN][-0Fork1][0][-0Br][0=Fork1][1][0=OH1][-0OH1][-C][=C][-Fork2][2][6][-CH1][=C][-Fork1][9][-O][-CH2][-C][=CH1][-CH1][=CH1][-CH1][=CH1][-Link1][4][-C][-C][-Fork1][1][-C][#N][=CH1][-N][-Fork1][4][-CH1][-Fork1][0][-CH3][-CH3][-C][=Fork1][0][=O][-C][=Link1][a][-Link2][1][6][-CH3][END]',
 '[BEGIN][-Fork2][1][d][-C][=CH1][-C][-Fork2][1][4][-OH1][0-C][-Fork1][f][-C][-Fork1][5][-O][-CH1][-Fork1][0][-CH3][-CH3][=C][-Fork1][0][-F][-CH1][=Link1][8][=O][-0F][-Fork1][0][-CH3][-CH3][END]',
 '[BEGIN][-CH1][=N][-NH1][-Fork1][2][-CH1][=Link1][3][0-CH1][-0Fork1][4][-0N][-Fork1][0][-CH3][-CH3][=O][END]',
 '[BEGIN][-0F